[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/04_mpc.ipynb)

# Part 4 — MPC-style planning

> **Control flow: a simulator.** The LLM proposes candidate plans; a forward model ranks them.

Model Predictive Control, borrowed from process control: at each step, look ahead, plan the next
*N* actions against a world model, **execute only the first**, then re-plan from the new state.

The agentic version swaps one box — the LLM proposes the candidates, and the simulator (not the
LLM) decides which is best:

```
state ──▶ plan (LLM, N candidates) ──▶ simulate (forward model) ──▶ act (first step only)
  ▲                                                                          │
  └──────────────────────── re-plan from the new state ◀────────────────────┘
```

The LLM contributes what it is good at: proposing plausible candidates from context.
The simulator contributes what it is good at: **being right**.

In [ ]:
# Setup -- same as 00_api_access.ipynb
import os, sys, time, json, re
if 'google.colab' in sys.modules:
    %pip install -U -q google-genai

from google import genai
from google.genai import types as gtypes
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint, solve_ivp
from scipy.optimize import minimize_scalar

plt.rcParams['figure.dpi'] = 100

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def generate_with_retry(*, contents, config=None, max_attempts=6):
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)


def llm_text(system_prompt, user_prompt, temperature=0.2):
    resp = generate_with_retry(
        contents=user_prompt,
        config=gtypes.GenerateContentConfig(
            system_instruction=system_prompt, temperature=temperature))
    return resp.text or ""


def run_agent(system_prompt, user_prompt, tool_schemas, tool_fns,
              max_steps=12, temperature=0.2, verbose=True):
    contents = [gtypes.Content(role="user",
                               parts=[gtypes.Part.from_text(text=user_prompt)])]
    cfg = gtypes.GenerateContentConfig(
        system_instruction=system_prompt,
        tools=[gtypes.Tool(function_declarations=tool_schemas)],
        temperature=temperature)

    transcript = []
    for step in range(max_steps):
        resp = generate_with_retry(contents=contents, config=cfg)
        parts = resp.candidates[0].content.parts or []
        contents.append(resp.candidates[0].content)

        calls = [p.function_call for p in parts if getattr(p, "function_call", None)]
        if not calls:
            text = "".join(getattr(p, "text", "") or "" for p in parts)
            transcript.append(("final", text))
            if verbose: print(f"[{step}] FINAL: {text[:160]}")
            return text, transcript

        obs = []
        for fc in calls:
            name, args = fc.name, dict(fc.args or {})
            if verbose: print(f"[{step}] CALL {name}({args})")
            try:
                result = tool_fns[name](**args)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            transcript.append((name, args, result))
            if verbose: print(f"[{step}]   -> {json.dumps(result)[:160]}")
            obs.append(gtypes.Part.from_function_response(name=name, response=result))
        contents.append(gtypes.Content(role="user", parts=obs))

    transcript.append(("final", "(max_steps reached)"))
    return "(max_steps reached)", transcript

print(f"Gemini client ready (model={MODEL}).")

### 4.1 The environment

A projectile with quadratic drag. The agent will only ever see `evaluate(angle) -> range`; it
does not get to look at the ODE.

$$\ddot{x} = -c_d |v|\dot{x}, \qquad \ddot{y} = -g - c_d|v|\dot{y}$$

with $v_0 = 50$ m/s, $c_d = 0.01$ m$^{-1}$, $g = 9.81$ m/s².

Without drag the optimum is exactly 45°. With drag it shifts lower — so the model's prior is
*almost* right, which is the interesting case.

In [ ]:
V0, CD, G = 50.0, 0.01, 9.81

def _rhs_drag(s, t):
    x, y, vx, vy = s
    v = np.sqrt(vx*vx + vy*vy)
    return [vx, vy, -CD*v*vx, -G - CD*v*vy]

def simulate_range(angle_deg):
    """Horizontal range (m) for a given launch angle. This is the expensive action."""
    a = np.deg2rad(angle_deg)
    s0 = [0.0, 0.0, V0*np.cos(a), V0*np.sin(a)]
    t = np.linspace(0, 20, 4001)
    sol = odeint(_rhs_drag, s0, t)
    y = sol[:, 1]
    hit = np.where((y[:-1] >= 0) & (y[1:] < 0))[0]
    if len(hit) == 0:
        return float('nan')
    i = hit[0]
    frac = y[i] / (y[i] - y[i+1])
    return float(sol[i, 0] + frac * (sol[i+1, 0] - sol[i, 0]))

# Ground truth, for scoring only. No agent gets to see this.
_res = minimize_scalar(lambda a: -simulate_range(a), bounds=(1, 89),
                       method='bounded', options={'xatol': 1e-4})
TRUE_OPT, TRUE_RANGE = float(_res.x), float(-_res.fun)
print(f"True optimum: {TRUE_OPT:.4f} deg  ->  {TRUE_RANGE:.4f} m")
print(f"(scipy.minimize_scalar needed {_res.nfev} evaluations to find it)")

angles = np.linspace(5, 85, 81)
ranges = [simulate_range(a) for a in angles]
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(angles, ranges, 'b-')
ax.axvline(TRUE_OPT, color='k', ls='--', lw=1, label=f'optimum {TRUE_OPT:.2f} deg')
ax.axvline(45, color='r', ls=':', lw=1, label='45 deg (no-drag prior)')
ax.set_xlabel("launch angle (deg)"); ax.set_ylabel("range (m)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 4.2 First, why the naive version is degenerate

An honest detour, because it is the mistake everyone makes on their first MPC agent.

MPC needs a forward model that is **cheaper than acting**. If you "simulate" a candidate angle by
calling `simulate_range` on it, you have already paid the full cost of the action — so
lookahead buys you nothing. You have built an expensive way to do a grid search.

**The fix:** the forward model has to be a genuine *surrogate*. Here, a quadratic least-squares
fit through the best few observations — essentially free, and accurate near the peak, which is
the only place accuracy matters.

In [ ]:
def fit_surrogate(obs, k=5):
    """Quadratic LSQ fit through the k best observations. This is the cheap forward model."""
    if len(obs) < 3:
        return None
    best = sorted(obs, key=lambda o: -o[1])[:max(k, 3)]
    a = np.array([o[0] for o in best], float)
    r = np.array([o[1] for o in best], float)
    if len(np.unique(a)) < 3:
        return None
    return np.polyfit(a, r, 2)

def surrogate_predict(coef, angle):
    return float(np.polyval(coef, angle))

def surrogate_vertex(coef):
    """Argmax of the fitted quadratic, if it opens downward."""
    A, B, _ = coef
    return None if A >= 0 else float(-B / (2 * A))

# Sanity check: three points is enough to locate the peak roughly.
demo = [(a, simulate_range(a)) for a in (20.0, 45.0, 70.0)]
c = fit_surrogate(demo)
print(f"3 observations -> surrogate vertex at {surrogate_vertex(c):.2f} deg "
      f"(true {TRUE_OPT:.2f} deg)")

### 4.3 The MPC loop

Read the loop for what the LLM is **not** allowed to do: it never decides which plan is best.
It generates hypotheses; the forward model adjudicates. That division is the entire pattern.

In [ ]:
def mpc_optimize(propose, evaluate, budget=12, n_candidates=6,
                 seeds=(20.0, 45.0, 70.0), verbose=True):
    """Plan -> simulate (surrogate) -> act (ONE real evaluation) -> re-plan."""
    obs = [(a, evaluate(a)) for a in seeds]
    while len(obs) < budget:
        coef = fit_surrogate(obs)                                  # 1. world model
        cands = [c for c in propose(obs, n_candidates, coef) if 0 < c < 90]   # 2. plan
        if not cands:
            print("  proposer returned no candidates in (0, 90); stopping early")
            break
        if coef is None:
            best = cands[0]
        else:
            best = max(cands, key=lambda a: surrogate_predict(coef, a))       # 3. simulate
        obs.append((best, evaluate(best)))                         # 4. act: one action only
        if verbose:
            print(f"  n={len(obs):2d}  proposed {len(cands)}  acted on {best:6.2f}"
                  f"  -> {obs[-1][1]:7.3f}")
    return obs

In [ ]:
def heuristic_proposer(obs, n, coef):
    """A non-LLM proposer: bracket the incumbent, plus the surrogate's vertex.
    Part 5 swaps in an LLM here; the loop itself does not change."""
    best_angle = max(obs, key=lambda o: o[1])[0]
    spread = max(1.0, 20.0 / (1 + len(obs)))
    cands = list(np.linspace(best_angle - spread, best_angle + spread, n - 1))
    v = surrogate_vertex(coef) if coef is not None else None
    cands.append(v if v is not None else best_angle)
    return cands

evals = {"n": 0}
def counted_eval(a):
    evals["n"] += 1
    return simulate_range(a)

print("MPC with the heuristic proposer (no LLM):")
obs_mpc = mpc_optimize(heuristic_proposer, counted_eval, budget=12)
a_mpc, r_mpc = max(obs_mpc, key=lambda o: o[1])
print(f"\nbest = {a_mpc:.4f} deg (error {abs(a_mpc-TRUE_OPT):.4f}), "
      f"range = {r_mpc:.4f} m, evaluations = {evals['n']}")

### 4.4 When to reach for MPC

**Use it when:**
- a wrong action is **expensive or irreversible** — burning budget, committing a mesh, running an experiment;
- you have a **cheap forward model you trust more than the LLM**;
- actions compose, so looking *N* steps ahead genuinely differs from looking one step ahead.

**Don't bother when:**
- simulating a plan costs about what executing it costs — then just execute and observe (§4.2);
- you have **no** forward model, in which case MPC degenerates into the LLM grading its own
  homework: slower than ReAct and equally wrong;
- the problem is small and convex. Use `scipy.optimize` and go home.

That last one is not a joke, and Part 5 measures it.

# Part 5 — Head-to-head: ReAct vs. MPC

Everything so far has been assertion. This part measures.

**The experiment.** Same problem (find the optimal launch angle), same evaluation budget (12
calls to `evaluate`), two architectures:

- **ReAct** — the model picks the next angle to try, one at a time, from the history.
- **MPC** — the model proposes 6 candidates; the *surrogate* picks which one to spend the
  evaluation on.

Plus two non-LLM baselines that keep everyone honest: a uniform grid at the same budget, and
`scipy.optimize.minimize_scalar`.

**Scoring:** absolute error in the located angle, at a fixed evaluation budget.

### 5.1 The ReAct optimizer

Three read-only tools and a budget. The model chooses its own search strategy.

In [ ]:
EVAL_LOG = []

def tool_evaluate(angle_deg: float):
    """Run the black-box projectile simulator and return the horizontal range."""
    if not (0 < angle_deg < 90):
        return {"error": f"angle must be in (0, 90) degrees; got {angle_deg}"}
    if len(EVAL_LOG) >= 12:
        return {"error": "evaluation budget exhausted; call propose_answer now"}
    r = simulate_range(angle_deg)
    EVAL_LOG.append({"angle_deg": float(angle_deg), "range": r})
    return {"angle_deg": angle_deg, "range": r, "evaluations_used": len(EVAL_LOG)}

def tool_history():
    """Return all previous (angle, range) evaluations."""
    return {"evaluations": EVAL_LOG, "count": len(EVAL_LOG)}

def tool_propose_answer(angle_deg: float, rationale: str = ""):
    """State your best estimate of the optimal launch angle and stop."""
    return {"proposed_optimum_deg": angle_deg, "rationale": rationale}

OPT_TOOL_FNS = {"evaluate": tool_evaluate, "history": tool_history,
                "propose_answer": tool_propose_answer}

def opt_schemas():
    return [
        gtypes.FunctionDeclaration(
            name="evaluate",
            description="Run the black-box projectile simulator for a launch angle in (0,90) "
                        "degrees and return the horizontal range in meters.",
            parameters={"type": "object",
                        "properties": {"angle_deg": {"type": "number"}},
                        "required": ["angle_deg"]}),
        gtypes.FunctionDeclaration(
            name="history",
            description="Return all previous (angle, range) evaluations.",
            parameters={"type": "object", "properties": {}, "required": []}),
        gtypes.FunctionDeclaration(
            name="propose_answer",
            description="State your best estimate of the optimal launch angle and stop.",
            parameters={"type": "object",
                        "properties": {"angle_deg": {"type": "number"},
                                       "rationale": {"type": "string"}},
                        "required": ["angle_deg"]}),
    ]

In [ ]:
REACT_SYSTEM = """You optimize a black-box function. It takes a launch angle in degrees,
strictly between 0 and 90, and returns a horizontal range in meters. Your goal is to locate the
angle that maximizes range, as precisely as you can.

Budget: at most 12 evaluate() calls. Spend them well -- coarse first, then refine around the
best region you have found. Use history() if you lose track. When your budget is spent or you
are confident, call propose_answer with your best estimate."""

react_angle = None
EVAL_LOG.clear()
ans, tr_react = run_agent(REACT_SYSTEM, "Find the optimal angle. Budget: 12 evaluations.",
                          tool_schemas=opt_schemas(), tool_fns=OPT_TOOL_FNS,
                          max_steps=20, verbose=True)
for e in reversed(tr_react):
    if e[0] == "propose_answer":
        react_angle = float(e[1]["angle_deg"]); break
if react_angle is None and EVAL_LOG:
    react_angle = max(EVAL_LOG, key=lambda d: d["range"])["angle_deg"]
REACT_LOG = list(EVAL_LOG)
print(f"\nReAct proposed {react_angle:.4f} deg using {len(REACT_LOG)} evaluations")

### 5.2 The MPC optimizer

Same budget. The model's *only* job is to propose candidates; the surrogate decides where the
evaluation goes.

In [ ]:
def parse_numbers(text, lo=0.0, hi=90.0):
    """Pull floats out of a model reply. Robust to prose, markdown fences, and stray commas."""
    out = []
    for m in re.finditer(r"-?\d+(?:\.\d+)?", text or ""):
        v = float(m.group())
        if lo < v < hi:
            out.append(v)
    return out

def llm_proposer(obs, n, coef):
    """The LLM proposes candidates. It does NOT get to pick the winner."""
    hist = ", ".join(f"({a:.3f} -> {r:.3f})" for a, r in sorted(obs)[-12:])
    txt = llm_text(
        "You propose candidate parameter values for an optimizer. "
        "Reply with ONLY a JSON list of numbers. No prose, no explanation.",
        f"Observed (launch_angle_deg -> range_m): {hist}\n\n"
        f"Propose {n} new candidate angles strictly between 0 and 90 that are most worth "
        f"testing next to maximize range. Favour the neighbourhood of the best observations, "
        f"but include at least one exploratory value. Reply with only a JSON list.",
        temperature=0.7)
    return parse_numbers(txt)[:n]

In [ ]:
mpc_evals = {"n": 0}
def mpc_counted_eval(a):
    mpc_evals["n"] += 1
    return simulate_range(a)

print("MPC run:")
obs_head = mpc_optimize(llm_proposer, mpc_counted_eval, budget=12, n_candidates=6)
mpc_angle, mpc_range = max(obs_head, key=lambda o: o[1])
print(f"\nMPC located {mpc_angle:.4f} deg using {mpc_evals['n']} evaluations")

### 5.3 The scoreboard

In [ ]:
# Baseline 1: uniform grid at the same budget.
grid = np.linspace(5, 85, 12)
grid_obs = [(a, simulate_range(a)) for a in grid]
grid_angle = max(grid_obs, key=lambda o: o[1])[0]

# Baseline 2: scipy, unrestricted budget.
scipy_angle, scipy_evals = TRUE_OPT, _res.nfev

rows = [("uniform grid",            grid_angle,  12),
        ("MPC (surrogate-ranked)",  mpc_angle,   mpc_evals["n"]),
        ("scipy.minimize_scalar",   scipy_angle, scipy_evals)]
if react_angle is not None:
    rows.insert(1, ("ReAct (model-chosen)", react_angle, len(REACT_LOG)))

print(f"{'method':<26} {'angle (deg)':>12} {'|error|':>10} {'evals':>7}")
print("-" * 58)
for name, ang, ne in rows:
    print(f"{name:<26} {ang:>12.4f} {abs(ang-TRUE_OPT):>10.4f} {ne:>7d}")
print("-" * 58)
print(f"{'truth':<26} {TRUE_OPT:>12.4f} {0.0:>10.4f} {'--':>7}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(angles, ranges, 'b-', alpha=0.5, lw=1, label='range(angle)')
ax.scatter([a for a, _ in grid_obs], [r for _, r in grid_obs],
           marker='x', s=45, c='gray', label='uniform grid (12)')
if REACT_LOG:
    ax.scatter([d["angle_deg"] for d in REACT_LOG], [d["range"] for d in REACT_LOG],
               marker='o', s=45, c='tab:orange', alpha=0.8,
               label=f'ReAct ({len(REACT_LOG)})')
ax.scatter([a for a, _ in obs_head], [r for _, r in obs_head],
           marker='s', s=45, c='tab:green', alpha=0.8,
           label=f'MPC ({mpc_evals["n"]})')
ax.axvline(TRUE_OPT, color='k', ls='--', lw=1, label=f'truth {TRUE_OPT:.2f} deg')
ax.set_xlabel("launch angle (deg)"); ax.set_ylabel("range (m)")
ax.set_title("Where each method spent its evaluations")
ax.set_ylim(min(ranges) - 2, max(ranges) + 2)
ax.legend(loc='lower center', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 5.4 Reading the result honestly

Look at *where the points are*, not just the final numbers. The grid spreads its budget evenly
over a region it already knows is uninteresting. MPC clusters almost everything within a few
degrees of the peak, because after three evaluations the surrogate already knows roughly where
the peak is.

**What this experiment does and does not show.**

- It **does** show that a cheap forward model converts a fixed budget into far more precision
  than either uniform sampling or step-by-step model judgment. That is the MPC thesis, and it
  holds here.
- It **does not** show that you should use an LLM for this problem. `scipy.minimize_scalar`
  finds the same answer with no model, no key, and no prompt. **If an agent cannot beat scipy on
  a 1-D smooth unimodal problem, that is worth knowing and worth saying out loud.**
- The LLM's contribution in the MPC run is *proposal diversity*, not decision quality. Swap
  `llm_proposer` for `heuristic_proposer` and compare — on this problem the difference is small,
  which tells you the surrogate is doing the work.

**So when is the LLM earning its place?** When the proposal step needs context that is not in the
data: physical priors, unit awareness, a plausible range for a parameter it has never seen,
constraints stated in prose. This projectile problem is small enough to see through — which is
exactly why it is a good place to build the intuition before you point the same machinery at a
problem where you cannot.

**Run it a few times.** The ReAct number will move around noticeably; MPC much less, because the
surrogate stabilizes it. Variance across runs *is* a result — a pattern that only works
sometimes has not been shown to work.